In [1]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 66.0 MB/s eta 0:00:00
--2026-04-14 20:02:03--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-04-14 20:02:03--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-14T20%3A39%3A34Z&rscd=attachment%3B+filename%3Dclou

In [2]:
%%writefile Ecotype.py
import streamlit as st
import numpy as np
import pickle
import os

# Load the trained model
model_path = '/content/drive/MyDrive/datascience/Mini Projects/EcoType: Forest Cover Classification/final_rf_model.pkl'
with open(model_path, "rb") as f:
    model = pickle.load(f)

# Load label encoder
label_encoder_path = "/content/drive/MyDrive/datascience/Mini Projects/EcoType: Forest Cover Classification/label_encoder.pkl"
with open(label_encoder_path, "rb") as f:
    label_encoder = pickle.load(f)

st.set_page_config(layout="wide")
st.set_page_config(
    page_title="Ecotype",  # This renames the browser tab
    page_icon="🌲"
)

st.title("🌲 Forest Cover Prediction") # This stays on the page
st.write("Enter the values for the following features to predict the forest cover type:")

# --- INPUT SECTION: Organized into two columns ---
col_in1, col_in2,col_in3, col_in4,col_in5 = st.columns(5)

with col_in1:
  Elevation = st.number_input("Elevation", min_value=1800, max_value=4000, value=2000)
  Wilderness_Area_1 = st.selectbox("Wilderness Area 1", [0, 1])
with col_in2:
  Horizontal_Distance_To_Roadways = st.number_input("Horizontal Distance To Roadways", min_value=0,max_value=7200, value=2000)
  Wilderness_Area_4 = st.selectbox("Wilderness Area 4", [0, 1])
with col_in3:
  Horizontal_Distance_To_Fire_Points = st.number_input("Horizontal Distance To Fire Points", min_value=0,max_value=7200, value=2000)
  Wilderness_Area_3 = st.selectbox("Wilderness Area 3", [0, 1])
with col_in4:
  Horizontal_Distance_To_Hydrology = st.number_input("Horizontal Distance To Hydrology", min_value=0,max_value=1400, value=1000)
  Soil_Type_12 =st.selectbox("Soil Type 12", [0, 1])
with col_in5:
  Vertical_Distance_To_Hydrology = st.number_input("Vertical Distance To Hydrology", min_value=-160,max_value=600, value=100)
  Soil_Type_23 =st.selectbox("Soil Type 23", [0, 1])

Aspect = st.slider("Aspect", min_value=0, max_value=360, value=90)
Hillshade_3pm= st.slider("Hillshade 3pm", min_value=0, max_value=248, value=100)
Hillshade_Noon = st.slider("Hillshade Noon", min_value=99, max_value=255, value=208)
Hillshade_9am = st.slider("Hillshade 9am", min_value=8, max_value=255, value=158)
Slope = st.slider("Slope", min_value=0, max_value=70, value=10)

#Arrange inputs into feature vector
features = np.array([[
  Elevation,
  Horizontal_Distance_To_Roadways,
  Horizontal_Distance_To_Fire_Points,
  Horizontal_Distance_To_Hydrology,
  Vertical_Distance_To_Hydrology,
  Aspect,
  Hillshade_3pm,
  Hillshade_Noon,
  Wilderness_Area_1,
  Hillshade_9am,
  Slope,
  Wilderness_Area_4,
  Wilderness_Area_3,
  Soil_Type_12,
  Soil_Type_23]])

# Define your image map based on your label_encoder classes
base_path = "/content/drive/MyDrive/datascience/Mini Projects/EcoType: Forest Cover Classification/"
image_map = {
    "Spruce/Fir": base_path + "Spruce.jpg",
    "Lodgepole Pine": base_path + "Lodgepole.JPG",
    "Ponderosa Pine": base_path + "Ponderosa.JPG",
    "Cottonwood/Willow": base_path + "Cottonwood.jpg",
    "Aspen": base_path + "Aspen.jpg",
    "Douglas-fir": base_path + "Douglas-fir.jpeg",
    "Krummholz": base_path + "Krummholz.jpg"
}

# Predict
if st.button("Predict Forest Cover Type"):
    prediction_encoded = model.predict(features)
    prediction_label = label_encoder.inverse_transform(prediction_encoded)[0]

    st.divider() # Adds a visual line

    res_col1, res_col2 = st.columns(2)

    with res_col1:
        st.success(f"Predicted Type: **{prediction_label}**")
        st.write("This area is likely dominated by this specific forest cover.")

    with res_col2:
        img_path = image_map.get(prediction_label)
        if img_path and os.path.exists(img_path):
            st.image(img_path, caption=f"Representative image of {prediction_label}", use_container_width=True)
        else:
            st.warning("Image not found in Drive. Check file names/paths.")


Writing Ecotype.py


In [3]:
!streamlit run /content/Ecotype.py &>/content/logs.txt &

In [5]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://foo-hazards-connecticut-ave.trycloudflare.com
